In [1]:
pip install sqlalchemy-utils 

Note: you may need to restart the kernel to use updated packages.Defaulting to user installation because normal site-packages is not writeable




[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: C:\Users\casat\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
pip install python-dotenv

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: C:\Users\casat\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd 
import os
import psycopg2
from sqlalchemy import create_engine
from sqlalchemy_utils import database_exists, create_database  
from dotenv import load_dotenv 
from pathlib import Path
import sys



Carga de datos de las tablas de Pedidos

In [ ]:

# Forzar a Python y PostgreSQL a ignorar las codificaciones locales de Windows
os.environ['PGCLIENTENCODING'] = 'utf-8'
os.environ['LC_ALL'] = 'C'
if sys.platform == "win32":
    # Fuerza a la consola de Windows a usar UTF-8 de forma nativa
    os.system('chcp 65001 > nul')

import pandas as pd
import psycopg2
from dotenv import load_dotenv
from sqlalchemy import create_engine

# Cargar las variables de entorno desde el archivo .env
load_dotenv()

# Obtener las variables de entorno para la conexión
db_user = os.getenv('db_user')
db_password = os.getenv('db_password')
db_host = os.getenv('db_host')
db_port = os.getenv('db_port')
db_name = os.getenv('db_name')

def asegurar_base_datos():
    
    # 2. SEGUNDO BLINDAJE: Forzar el encoding en las opciones nativas de conexión

    try:
        conn = psycopg2.connect(
            host=db_host,
            port=db_port,
            user=db_user,
            password=db_password,
            database='postgres',  
            client_encoding='utf-8', # Forzamos UTF-8 aquí
            options="-c client_encoding=utf8" # Forzamos parámetros internos del servidor
        )
    except UnicodeDecodeError:
        # Si aun así falla por temas de caracteres, usamos 'latin1' que acepta cualquier byte de Windows
        conn = psycopg2.connect(
            host=db_host,
            port=db_port,
            user=db_user,
            password=db_password,
            database='postgres',  
            client_encoding='latin1'
        )
        
    conn.autocommit = True 
    cursor = conn.cursor()

    # Buscamos si la base de datos ya existe en el servidor
    cursor.execute(f"SELECT 1 FROM pg_database WHERE datname='{db_name}'")
    exists = cursor.fetchone()

    if not exists:
        print(f"La base de datos '{db_name}' no existe. Creando...")
        cursor.execute(f"CREATE DATABASE {db_name}")
        print(f"Base de datos '{db_name}' creada exitosamente.")
    else:
        print(f"Conexión verificada: La base de datos '{db_name}' ya existe y está lista.")

    cursor.close()
    conn.close()

def run_etl():
    # Asegurar que la base de datos exista
    asegurar_base_datos()

    # Crear la conexión a la base de datos definitiva
    url_conexion = f'postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}'
    engine = create_engine(url_conexion)

    # Extraer los archivos de la carpeta origen
    carpeta_origen = r"C:\Users\casat\OneDrive\Documentos\Anibal personal\ThePower Business School\Power BI\Caso Practico_Datos\Pedidos"
    if not os.path.exists(carpeta_origen):
        os.makedirs(carpeta_origen)
        print(f"La carpeta '{carpeta_origen}' no existe.")
        return
    
    # Buscar los archivos que terminan en .xlsx dentro de la carpeta origen
    archivos = [f for f in os.listdir(carpeta_origen) if f.endswith('.xlsx')]

    if archivos:
        df_list = [pd.read_excel(os.path.join(carpeta_origen, a)) for a in archivos]
        df_final = pd.concat(df_list, ignore_index=True)

        # Transformar los datos (limpieza)
        df_final.columns = [c.lower().replace(' ', '_').strip() for c in df_final.columns]
        
        # Cargar los datos en la base de datos
        print(f"Cargando datos en la base de datos en {db_name} ...")
        df_final.to_sql('pedidos', engine, if_exists='replace', index=False)
        print("Datos cargados exitosamente.")
    else:
        print(f"No se encontraron archivos XLSX en la carpeta '{carpeta_origen}'.")

if __name__ == "__main__":
    run_etl()

Conexión verificada: La base de datos 'ventasdb' ya existe y está lista.
Cargando datos en la base de datos en ventasdb ...
Datos cargados exitosamente.


Carga de tabla Clientes

In [ ]:


# Cargar las variables de entorno
load_dotenv()

# Obtener las variables de entorno para la conexión
db_user = os.getenv('db_user')
db_password = os.getenv('db_password')
db_host = os.getenv('db_host')
db_port = os.getenv('db_port')
db_name = os.getenv('db_name')

def conexion_db():
    # Conexión inicial a la base genérica 'postgres' para verificar/crear la base de datos destino
    conn = psycopg2.connect(
        host=db_host,
        port=db_port,
        user=db_user,
        password=db_password,
        database='postgres',  
        client_encoding='utf-8', 
        options="-c client_encoding=utf8" 
    )

    conn.autocommit = True
    cursor = conn.cursor()

    # Verificar si la base de datos ya existe
    cursor.execute(f"SELECT 1 FROM pg_database WHERE datname='{db_name}'")
    exists = cursor.fetchone()

    if not exists:
        print(f'La base de datos "{db_name}" no existe. Creando....')
        cursor.execute(f"CREATE DATABASE {db_name}")
        print(f'Base de datos "{db_name}" creada exitosamente.')
    else:
        print(f'La base de datos "{db_name}" ya existe.')
        
    cursor.close()
    conn.close()

def run_etl():
    # 1. Asegurar que la base de datos exista
    conexion_db()

    # 2. Crear la conexión a la base de datos DEFINITIVA (usando db_name)
    url_conexion = f'postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}'
    engine = create_engine(url_conexion)   

    # 3. Definir rutas usando pathlib de forma correcta
    carpeta_origen = Path(r"C:\Users\casat\OneDrive\Documentos\Anibal personal\ThePower Business School\Power BI\Caso Practico_Datos")
    nombre_archivo = "Clientes.xlsx"
    ruta_tabla = carpeta_origen / nombre_archivo

    # 4. Proceso de Extracción y Transformación
    if ruta_tabla.exists():
        print(f"Archivo encontrado: {ruta_tabla}")
        
        # Cargar el archivo Excel
        df_clientes = pd.read_excel(ruta_tabla)
        
        # Transformación 1: Forzar minúsculas, cambiar espacios por guiones bajos y limpiar extremos
        df_clientes.columns = [c.lower().replace(' ', '_').strip() for c in df_clientes.columns]
        
        # Transformación 2: Renombrar columnas específicas de manera segura con Pandas
         # Nota: Mapeamos los nombres originales (ya en minúsculas por el paso anterior) a los nuevos nombres limpios.
        columnas_nuevas = {
            'Clientes.Cliente_ID': 'cliente_id',  
            'Clientes.Nombre': 'nombre', 
            'Clientes.Contacto': 'contacto',  
            'Clientes.Ciudad': 'ciudad', 
            'Clientes.Pais': 'pais',
            'Clientes.Division': 'division', 
            'Clientes.Direccion': 'direccion', 
            'Clientes.CodigoPostal': 'codigo_postal'
        }
        df_clientes = df_clientes.rename(columns=columnas_nuevas)

        # 5. Carga de datos en la base de datos
        print(f"Cargando datos en la tabla 'clientes' dentro de la base de datos '{db_name}' ...")
        # 'replace' sobreescribe la tabla si ya existe. Cambiar a 'append' si solo quieres sumar filas.
        df_clientes.to_sql('clientes', engine, if_exists='replace', index=False)
        print("¡Datos cargados exitosamente!")
        
    else:
        print(f"ERROR: No se encontró el archivo '{nombre_archivo}' en la carpeta '{carpeta_origen}'.")
        return

if __name__ == "__main__":
    run_etl()

La base de datos "ventasdb" ya existe.
Archivo encontrado: C:\Users\casat\OneDrive\Documentos\Anibal personal\ThePower Business School\Power BI\Caso Practico_Datos\Clientes.xlsx
Cargando datos en la tabla 'clientes' dentro de la base de datos 'ventasdb' ...
¡Datos cargados exitosamente!


In [ ]:
#CARGA DE LA TABLA PRODUCTOS DESDE EXCEL A POSTGRESQL

# Cargar las variables de entorno
load_dotenv()

# Obtener las variables de entorno para la conexión
db_user = os.getenv('db_user')
db_password = os.getenv('db_password')
db_host = os.getenv('db_host')
db_port = os.getenv('db_port')
db_name = os.getenv('db_name')

def conexion_db():
    # Conexión inicial a la base genérica 'postgres' para verificar/crear la base de datos destino
    conn = psycopg2.connect(
        host=db_host,
        port=db_port,
        user=db_user,
        password=db_password,
        database='postgres',  
        client_encoding='utf-8', 
        options="-c client_encoding=utf8" 
    )

    conn.autocommit = True
    cursor = conn.cursor()

    # Verificar si la base de datos ya existe
    cursor.execute(f"SELECT 1 FROM pg_database WHERE datname='{db_name}'")
    exists = cursor.fetchone()

    if not exists:
        print(f'La base de datos "{db_name}" no existe. Creando....')
        cursor.execute(f"CREATE DATABASE {db_name}")
        print(f'Base de datos "{db_name}" creada exitosamente.')
    else:
        print(f'La base de datos "{db_name}" ya existe.')
        
    cursor.close()
    conn.close()

def run_etl():
    # 1. Asegurar que la base de datos exista
    conexion_db()

    # 2. Crear la conexión a la base de datos DEFINITIVA (usando db_name)
    url_conexion = f'postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}'
    engine = create_engine(url_conexion)   

    # 3. Definir rutas usando pathlib de forma correcta
    carpeta_origen = Path(r"C:\Users\casat\OneDrive\Documentos\Anibal personal\ThePower Business School\Power BI\Caso Practico_Datos")
    nombre_archivo = "Productos.xlsx"
    ruta_tabla = carpeta_origen / nombre_archivo

    # 4. Proceso de Extracción y Transformación
    if ruta_tabla.exists():
        print(f"Archivo encontrado: {ruta_tabla}")
        
        # Cargar el archivo Excel
        df_productos = pd.read_excel(ruta_tabla)
        
        # Transformación 1: Forzar minúsculas, cambiar espacios por guiones bajos y limpiar extremos
        df_productos.columns = [c.lower().replace(' ', '_').strip() for c in df_productos.columns]
        
        # Transformación 2: Renombrar columnas específicas de manera segura con Pandas
         # Nota: Mapeamos los nombres originales (ya en minúsculas por el paso anterior) a los nuevos nombres limpios.
		

        columnas_nuevas = {
            'Productos.Producto_ID': 'producto_id',
            'Productos.CantidadPorUnidad': 'cantidad_por_unidad',  
            'Productos.Nombre': 'nombre',     
            'Productos.PrecioProducto': 'precio',
            'Productos.CosteProducto': 'coste_producto',
            'Productos.CantidadEnStock': 'stock', 
            'Productos.Categoria_ID': 'categoria', 
            'Productos.Proveedor': 'proveedor',
            'Productos.CosteUnitario': 'coste_unitario',
            'Productos.CantidadEnPedidos': 'cantidad_en_pedidos',
            'Productos.PrecioUnitario': 'precio_unitario'
        }
        df_productos = df_productos.rename(columns=columnas_nuevas)

        # 5. Carga de datos en la base de datos
        print(f"Cargando datos en la tabla 'productos' dentro de la base de datos '{db_name}' ...")
        # 'replace' sobreescribe la tabla si ya existe. Cambiar a 'append' si solo quieres sumar filas.
        df_productos.to_sql('productos', engine, if_exists='replace', index=False)
        print("¡Datos cargados exitosamente!")
        
    else:
        print(f"ERROR: No se encontró el archivo '{nombre_archivo}' en la carpeta '{carpeta_origen}'.")
        return

if __name__ == "__main__":
    run_etl()

La base de datos "ventasdb" ya existe.
Archivo encontrado: C:\Users\casat\OneDrive\Documentos\Anibal personal\ThePower Business School\Power BI\Caso Practico_Datos\Productos.xlsx
Cargando datos en la tabla 'productos' dentro de la base de datos 'ventasdb' ...
¡Datos cargados exitosamente!


In [ ]:
            'Vendedores.Vendedor_ID': 'vendedor_id',
            'Vendedores.Nombre': 'nombre',
            'Vendedores.Apellido': 'apellido',
            'Vendedores.FechaNacimiento': 'fecha_nacimiento',
            'Vendedores.FechaContratacion': 'fecha_contratacion',
            'Vendedores.FechaBaja': 'fecha_baja',
            'Vendedores.Oficina_ID': 'oficina_id',
            'Vendedores.Puesto': 'puesto'

In [8]:
#CARGA DE LA TABLA VENDEDORES DESDE EXCEL A POSTGRESQL

# Cargar las variables de entorno
load_dotenv()

# Obtener las variables de entorno para la conexión
db_user = os.getenv('db_user')
db_password = os.getenv('db_password')
db_host = os.getenv('db_host')
db_port = os.getenv('db_port')
db_name = os.getenv('db_name')

def conexion_db():
    # Conexión inicial a la base genérica 'postgres' para verificar/crear la base de datos destino
    conn = psycopg2.connect(
        host=db_host,
        port=db_port,
        user=db_user,
        password=db_password,
        database='postgres',  
        client_encoding='utf-8', 
        options="-c client_encoding=utf8" 
    )

    conn.autocommit = True
    cursor = conn.cursor()

    # Verificar si la base de datos ya existe
    cursor.execute(f"SELECT 1 FROM pg_database WHERE datname='{db_name}'")
    exists = cursor.fetchone()

    if not exists:
        print(f'La base de datos "{db_name}" no existe. Creando....')
        cursor.execute(f"CREATE DATABASE {db_name}")
        print(f'Base de datos "{db_name}" creada exitosamente.')
    else:
        print(f'La base de datos "{db_name}" ya existe.')
        
    cursor.close()
    conn.close()

def run_etl():
    # 1. Asegurar que la base de datos exista
    conexion_db()

    # 2. Crear la conexión a la base de datos DEFINITIVA (usando db_name)
    url_conexion = f'postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}'
    engine = create_engine(url_conexion)   

    # 3. Definir rutas usando pathlib de forma correcta
    carpeta_origen = Path(r"C:\Users\casat\OneDrive\Documentos\Anibal personal\ThePower Business School\Power BI\Caso Practico_Datos")
    nombre_archivo = "Vendedores.xlsx"
    ruta_tabla = carpeta_origen / nombre_archivo

    # 4. Proceso de Extracción y Transformación
    if ruta_tabla.exists():
        print(f"Archivo encontrado: {ruta_tabla}")
        
        # Cargar el archivo Excel
        df_vendedores = pd.read_excel(ruta_tabla)
        
        # Transformación 1: Forzar minúsculas, cambiar espacios por guiones bajos y limpiar extremos
        df_vendedores.columns = [c.lower().replace(' ', '_').strip() for c in df_vendedores.columns]
        
        # Transformación 2: Renombrar columnas específicas de manera segura con Pandas
         # Nota: Mapeamos los nombres originales (ya en minúsculas por el paso anterior) a los nuevos nombres limpios.
		

        columnas_nuevas = {
            'Vendedores.Vendedor_ID': 'vendedor_id',
            'Vendedores.Nombre': 'nombre',
            'Vendedores.Apellido': 'apellido',
            'Vendedores.FechaNacimiento': 'fecha_nacimiento',
            'Vendedores.FechaContratacion': 'fecha_contratacion',
            'Vendedores.FechaBaja': 'fecha_baja',
            'Vendedores.Oficina_ID': 'oficina_id',
            'Vendedores.Puesto': 'puesto'
        }
        df_vendedores = df_vendedores.rename(columns=columnas_nuevas)

        # 5. Carga de datos en la base de datos
        print(f"Cargando datos en la tabla 'vendedores' dentro de la base de datos '{db_name}' ...")
        # 'replace' sobreescribe la tabla si ya existe. Cambiar a 'append' si solo quieres sumar filas.
        df_vendedores.to_sql('vendedores', engine, if_exists='replace', index=False)
        print("¡Datos cargados exitosamente!")
        
    else:
        print(f"ERROR: No se encontró el archivo '{nombre_archivo}' en la carpeta '{carpeta_origen}'.")
        return

if __name__ == "__main__":
    run_etl()

La base de datos "ventasdb" ya existe.
Archivo encontrado: C:\Users\casat\OneDrive\Documentos\Anibal personal\ThePower Business School\Power BI\Caso Practico_Datos\Vendedores.xlsx
Cargando datos en la tabla 'vendedores' dentro de la base de datos 'ventasdb' ...
¡Datos cargados exitosamente!


In [ ]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

# Cargar variables de entorno
load_dotenv()

db_user = os.getenv("db_user")
db_password = os.getenv("db_password")
db_host = os.getenv("db_host")
db_port = os.getenv("db_port")
db_name = os.getenv("db_name")


def run_etl():

    # Conexión directamente a la base de datos existente
    url_conexion = (
        f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
    )

    engine = create_engine(url_conexion)

    # Ruta del archivo
    carpeta_origen = Path(
        r"C:\Users\casat\OneDrive\Documentos\Anibal personal\ThePower Business School\Power BI\Caso Practico_Datos"
    )

    nombre_archivo = "Vendedores.xlsx"
    ruta_tabla = carpeta_origen / nombre_archivo

    if not ruta_tabla.exists():
        print(f"No se encontró el archivo {ruta_tabla}")
        return

    print(f"Archivo encontrado: {ruta_tabla}")

    # Leer Excel
    df_vendedores = pd.read_excel(ruta_tabla)

    # Limpiar nombres de columnas
    df_vendedores.columns = (
        df_vendedores.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace(".", "_")
    )

    # Renombrar columnas
    columnas_nuevas = {
        "vendedores_vendedor_id": "vendedor_id",
        "vendedores_nombre": "nombre",
        "vendedores_apellido": "apellido",
        "vendedores_fechanacimiento": "fecha_nacimiento",
        "vendedores_fechacontratacion": "fecha_contratacion",
        "vendedores_fechabaja": "fecha_baja",
        "vendedores_oficina_id": "oficina_id",
        "vendedores_puesto": "puesto",
    }

    df_vendedores.rename(columns=columnas_nuevas, inplace=True)

    print("Creando la tabla 'vendedores' (si no existe)...")

    df_vendedores.to_sql(
        name="vendedores",
        con=engine,
        if_exists="append",   # crea la tabla si no existe
        index=False
    )

    print("Proceso finalizado correctamente.")


if __name__ == "__main__":
    run_etl()